In [0]:
from pyspark.sql import functions as F

catalog = "dbw_company_risk"
snapshot = F.lit("2026-10-01").cast("date")


def blank_to_null(col):
    """Turn empty text into a proper missing value."""
    return F.when(F.trim(col) == "", None).otherwise(col)


def uk_date(col_name):
    """Convert day/month/year text to a date; invalid values become null."""
    return F.expr(f"try_to_timestamp({col_name}, 'dd/MM/yyyy')").cast("date")


status = F.upper(F.col("companystatus"))
status_group = (
    F.when(status == "ACTIVE", "Active")
     .when(status.contains("STRIKE OFF"), "Proposal to strike off")
     .when(status.contains("LIQUIDATION"), "Liquidation")
     .when(status.contains("VOLUNTARY ARRANGEMENT"), "Voluntary arrangement")
     .when(status.contains("ADMINISTRAT") | status.contains("RECEIVER"), "Administration or receivership")
     .otherwise("Other")
)

silver = (
    spark.table(f"{catalog}.bronze.companies_raw")
    .select(
        F.trim("companynumber").alias("company_number"),
        F.col("companyname").alias("company_name"),
        F.col("companycategory").alias("company_type"),
        F.col("companystatus").alias("status_raw"),
        status_group.alias("status_group"),
        blank_to_null(F.regexp_extract("siccodesictext_1", r"^(\d+)", 1)).alias("sic_code"),
        blank_to_null(F.regexp_extract("siccodesictext_1", r"^\d+ - (.*)$", 1)).alias("sic_description"),
        F.col("regaddressposttown").alias("post_town"),
        F.upper("regaddresspostcode").alias("postcode"),
        blank_to_null(F.regexp_extract(F.upper("regaddresspostcode"), r"^([A-Z]{1,2})", 1)).alias("postcode_area"),
        uk_date("incorporationdate").alias("incorporation_date"),
        uk_date("accountsnextduedate").alias("accounts_next_due"),
        uk_date("accountslastmadeupdate").alias("accounts_last_made_up"),
        uk_date("confstmtnextduedate").alias("confirmation_next_due"),
        F.col("accountsaccountcategory").alias("accounts_category"),
        F.expr("coalesce(try_cast(mortgagesnummortoutstanding AS INT), 0)").alias("charges_outstanding"),
        F.col("_source_file"),
        F.col("_ingested_at"),
    )
    .withColumn("snapshot_date", snapshot)
    .withColumn("company_age_years",
                F.round(F.datediff("snapshot_date", "incorporation_date") / 365.25, 1))
    .withColumn("accounts_overdue",
                F.coalesce(F.col("accounts_next_due") < F.col("snapshot_date"), F.lit(False)))
    .withColumn("confirmation_overdue",
                F.coalesce(F.col("confirmation_next_due") < F.col("snapshot_date"), F.lit(False)))
)

(silver.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.companies"))

print(f"Silver rows: {spark.table(f'{catalog}.silver.companies').count():,}")

In [0]:
%sql
SELECT status_group,
       COUNT(*) AS companies,
       SUM(CAST(accounts_overdue AS INT)) AS accounts_overdue
FROM dbw_company_risk.silver.companies
GROUP BY status_group
ORDER BY companies DESC